<a href="https://colab.research.google.com/github/Ayushi1303/IN126001402_Data_Science-_Internship-February--2026/blob/main/Task_5_Fine_Tuning_BERT_for_POS_Tagging_and_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets seqeval -q

In [3]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset, DatasetDict
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

In [6]:
from google.colab import files

# Upload files manually
uploaded = files.upload()

# After upload, filenames will be available
train_file = "train.txt"
val_file = "valid.txt"
test_file = "test.txt"


def read_conll_file(filepath):
    sentences = []
    labels = []

    words = []
    chunk_tags = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(chunk_tags)
                    words = []
                    chunk_tags = []
            else:
                parts = line.strip().split()

                # Skip DOCSTART lines
                if parts[0] == "-DOCSTART-":
                    continue

                if len(parts) >= 3:
                    words.append(parts[0])
                    chunk_tags.append(parts[2])  # Chunk column

    return sentences, labels


train_texts, train_labels = read_conll_file(train_file)
val_texts, val_labels = read_conll_file(val_file)
test_texts, test_labels = read_conll_file(test_file)

print("Sample Sentence:", train_texts[0])
print("Sample Labels:", train_labels[0])

Saving test.txt to test.txt
Saving valid.txt to valid.txt
Saving train.txt to train.txt
Sample Sentence: ['He', 'reckons', 'the', 'current', 'account', 'deficit', 'will', 'narrow', 'to', 'only', '#', '1', '.']
Sample Labels: ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'I-NP', 'I-NP', 'B-VP', 'I-VP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'O']


In [7]:
train_dataset = Dataset.from_dict({
    "tokens": train_texts,
    "labels": train_labels
})

val_dataset = Dataset.from_dict({
    "tokens": val_texts,
    "labels": val_labels
})

test_dataset = Dataset.from_dict({
    "tokens": test_texts,
    "labels": test_labels
})

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1
    })
})

In [8]:
all_labels = []
for split in ["train", "validation", "test"]:
    for doc_labels in dataset[split]["labels"]:
        all_labels.extend(doc_labels)

label_list = sorted(list(set(all_labels)))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)

['B-NP', 'B-PP', 'B-VP', 'I-NP', 'I-VP', 'O']


In [9]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100) # Assign -100 for subsequent subword tokens

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [11]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

In [14]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.839108,0.000000,0.000000,0.000000
2,No log,1.830209,0.000000,0.000000,0.000000


TrainOutput(global_step=2, training_loss=1.7168819904327393, metrics={'train_runtime': 1.313, 'train_samples_per_second': 1.523, 'train_steps_per_second': 1.523, 'total_flos': 8166409344.0, 'train_loss': 1.7168819904327393, 'epoch': 2.0})

In [16]:
trainer.evaluate()

{'eval_loss': 1.8302093744277954,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_runtime': 0.0414,
 'eval_samples_per_second': 24.139,
 'eval_steps_per_second': 24.139,
 'epoch': 2.0}

In [17]:
sentence = "John works at Google in California"

tokens = sentence.split()

tokenizer_output = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in tokenizer_output.items()}

outputs = model(**inputs).logits
predictions = np.argmax(outputs.detach().cpu().numpy(), axis=2)

word_ids = tokenizer_output.word_ids()
predicted_labels = []

for idx, word_id in enumerate(word_ids):
    if word_id is not None:
        predicted_labels.append(id2label[predictions[0][idx]])

print(list(zip(tokens, predicted_labels)))

[('John', 'I-NP'), ('works', 'I-NP'), ('at', 'I-VP'), ('Google', 'I-VP'), ('in', 'I-VP'), ('California', 'I-NP')]


Comparison:
POS Tagging vs Chunking POS Tagging identifies the grammatical role of each word (e.g., noun, verb).
Chunking groups words into meaningful phrases (e.g., noun phrase, verb phrase).

Key Difference:
POS Tagging → Word-level classification (simpler) Chunking → Phrase-level grouping (more complex)

Difficulty: POS Tagging: Easy

Chunking: Medium Observation: Chunking provides better structural understanding of sentences compared to POS tagging.

Conclusion In this project, we successfully fine-tuned a DistilBERT model for chunking using token classification.

Key Achievements: Performed tokenization and label alignment Trained transformer model Evaluated using seqeval metrics Performed inference on custom sentence

Challenges:
Handling subword tokenization
Aligning labels correctly

Final Insight:
Transformer models like BERT are highly effective for sequence labeling tasks such as chunking.